# Description

W tym notatniku przeprowadzane są wszelkie eksperymenty, zarówno dla autoenkodera wariacyjnego i nie wariacyjnego, dla wszystkich członów funkcji straty, w wersji z douczaniem i bez (łącznie 12 eksperymentów)

# Imports

In [26]:
%load_ext autoreload
%autoreload 2
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import RichProgressBar
import yaml
import sys
import os
import tqdm
import wandb
import json

sys.path.append('../')  # Dodajemy katalog wyżej, żeby src był widoczny
from src.models.KlejdaGAE.GraphAutoencoder import GraphAutoencoder
from src.models.KlejdaGAE.VariationalGraphAutoencoder import VariationalGraphAutoencoder
from utils.FramsticksGraphDataset import FramsticksGraphDataset
from pyprojroot import here

current_dir = os.getcwd()
framspy_path = os.path.abspath(os.path.join(current_dir, '..', 'external', 'framspy'))
if framspy_path not in sys.path:
	sys.path.insert(0, framspy_path)
from FramsticksLib import FramsticksLib
from deap import tools, algorithms
import yaml
from src.deap.deap_setup import prepare_native_toolbox, prepare_cmaes_toolbox
from src.deap.constraints import is_feasible_fitness_criteria
from src.deap.save_and_load_results import save_genotypes_json
from utils.FramsticksPostProcessor import FramsticksPostProcessor
from utils.FramsticksGraphDataset import FramsticksGraphDataset
from src.deap.AutoencoderEvaluator import AutoencoderEvaluator
import numpy as np
import frams

import time

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Pre-processing

In [27]:
project_dir = here()
# Przygotowanie checkpointów nauczonych autoenkoderów
checkpoints_dir = project_dir / 'notebooks' / 'checkpoints' / 'final_checkpoints' / 'klejda'
checkpoint_gae = torch.load(checkpoints_dir / 'gae.ckpt')
checkpoint_vgae = torch.load(checkpoints_dir / 'vgae.ckpt')

In [28]:
# Przygotowanie konfiguracji dla gae
configs_dir = project_dir / 'configs'
config_gae_path = configs_dir / 'klejda_gae_config.yaml'
config_vgae_path = configs_dir / 'klejda_vgae_config.yaml'
with open(config_gae_path) as f:
	config_gae = yaml.safe_load(f)
with open(config_vgae_path) as f:
	config_vgae = yaml.safe_load(f)

In [18]:
# # Przygotowanie środowiska Framsticks oraz DEAP
# with open("../configs/final_evolution_config.yaml", 'r') as f:
# 	evolution_config = yaml.safe_load(f)
# frams.init(
#     evolution_config['frams_path']
# )
# frams_lib = FramsticksLib(evolution_config['frams_path'], evolution_config['frams_lib'], evolution_config['sim_file'])
#
# toolbox = prepare_native_toolbox(frams_lib, evolution_config)
# # TODO: TO dodać przed uruchomieniem eksperymentu
# # toolbox.register("mutate", autoencoder_mutate, gae)
# pop = toolbox.population(n=evolution_config['pop_size'])
# hof = tools.HallOfFame(evolution_config['hof_size'])
#
# stats = tools.Statistics(lambda ind: ind.fitness.values)
# filter_feasible = lambda func, criteria: func(list(filter(is_feasible_fitness_criteria, criteria)))
# stats.register("min", lambda fit: filter_feasible(np.min, fit))
# stats.register("avg", lambda fit: filter_feasible(np.mean, fit))
# stats.register("max", lambda fit: filter_feasible(np.max, fit))
#
# postProcessingGaeResult = FramsticksPostProcessor()

In [29]:
# Wersja CMA-ES
with open("../configs/final_evolution_config.yaml", 'r') as f:
	evolution_config = yaml.safe_load(f)
frams.init(
	evolution_config['frams_path']
)
frams_lib = FramsticksLib(evolution_config['frams_path'], evolution_config['frams_lib'], evolution_config['sim_file'])
toolbox = prepare_cmaes_toolbox(frams_lib, evolution_config)


hof = tools.HallOfFame(evolution_config['hof_size'])
stats = tools.Statistics(lambda ind: ind.fitness.values)
filter_feasible = lambda func, criteria: func(list(filter(is_feasible_fitness_criteria, criteria)))
stats.register("min", lambda fit: filter_feasible(np.min, fit))
stats.register("avg", lambda fit: filter_feasible(np.mean, fit))
stats.register("max", lambda fit: filter_feasible(np.max, fit))

Using Framsticks version: 5.5
Home (writable) dir     : C:\Users\witek\PycharmProjects\Magisterka\external\Framsticks55\data
Resources dir           : C:\Users\witek\PycharmProjects\Magisterka\external\Framsticks55\data

Using Framsticks version: 5.5
Home (writable) dir     : C:\Users\witek\PycharmProjects\Magisterka\external\Framsticks55\data
Resources dir           : C:\Users\witek\PycharmProjects\Magisterka\external\Framsticks55\data

Available objects: ['CheckpointEvent', 'Collision', 'CrCollision', 'Creature', 'CreatureSettings', 'CreatureSignals', 'CreatureSnapshot', 'Dictionary', 'ExpProperties', 'ExpState', 'ExtValue', 'File', 'FunctionReference', 'GenMan', 'GenManStats', 'GenePool', 'GenePools', 'Geno', 'GenoConverters', 'Genotype', 'Interface', 'Joint', 'Loader', 'Math', 'MechJoint', 'MechPart', 'MessageCatcher', 'Model', 'ModelGeometry', 'ModelSymmetry', 'Neuro', 'NeuroClass', 'NeuroClassLibrary', 'NeuroDef', 'NeuroSignals', 'NeuronsSimEnabled', 'ODE', 'Orient', 'Part', 'Pop

# Experiments

## GAE

### Trained once

In [30]:
gae_non_cyclic = GraphAutoencoder(config=config_gae, frams_module=frams).double()
gae_non_cyclic.load_state_dict(checkpoint_gae['state_dict'])
gae_non_cyclic.eval()
evaluator = AutoencoderEvaluator(gae_non_cyclic, frams_lib,evolution_config['opt_criteria'], evolution_config)
toolbox.register("evaluate", evaluator)

pop, log = algorithms.eaGenerateUpdate(
	 toolbox,
	ngen = evolution_config['generations'],
	stats=stats,
	halloffame=hof,
	verbose=True
)

print(f"\nNajlepszy fitness w HoF: {hof[0].fitness.values[0]}")
save_genotypes_json(evolution_config["result_filepath"], hof)

gen	nevals	min  	avg      	max     
0  	120   	-0.01	0.0374896	0.140898
1  	120   	-0.01	0.0205134	0.0947125
2  	120   	-0.01	0.030998 	0.155305 
3  	120   	-0.01	0.0369524	0.172196 
4  	120   	-0.01	0.0580848	0.30606  
5  	120   	-0.01	0.0229505	0.172007 
6  	120   	-0.01	0.0551154	0.20282  
7  	120   	-0.01	0.0873842	0.195208 
8  	120   	-0.01	0.128951 	0.199809 
9  	120   	-0.01	0.150471 	0.199085 
10 	120   	-0.01	0.177454 	0.204734 
11 	120   	0.00931553	0.178351 	0.254167 
12 	120   	0.0988484 	0.192178 	0.267152 
13 	120   	0.14933   	0.199129 	0.2708   
14 	120   	0.0914856 	0.202837 	0.271915 
15 	120   	-0.01     	0.220272 	0.272334 
16 	120   	0.150415  	0.238122 	0.27267  
17 	120   	0.144699  	0.25132  	0.273221 
18 	120   	0.149435  	0.248281 	0.273398 
19 	120   	0.0684654 	0.256365 	0.273845 
20 	120   	0.151767  	0.257319 	0.274102 
21 	120   	0.00514387	0.256911 	0.274635 
22 	120   	0.136899  	0.261432 	0.274765 
23 	120   	0.185154  	0.269299 	0.274955 
24 	120   	0

### Continual training

## VGAE

### Trained once

In [41]:
vgae_non_cyclic = VariationalGraphAutoencoder(config=config_vgae, frams_module=frams).double()
vgae_non_cyclic.load_state_dict(checkpoint_vgae['state_dict'])
vgae_non_cyclic.eval()
toolbox.register("mutate", autoencoder_mutate, vgae_non_cyclic)

pop, log = algorithms.eaSimple(
	pop, toolbox,
	cxpb=evolution_config['p_xov'], mutpb=evolution_config['p_mut'],
	ngen=evolution_config['generations'], stats=stats, halloffame=hof, verbose=True
)

print(f"\nNajlepszy fitness w HoF: {hof[0].fitness.values[0]}")
save_genotypes_json(evolution_config["result_filepath"], hof)

gen	nevals	min  	avg  	max  
0  	120   	-0.01	-0.01	-0.01
1  	112   	-0.01	-0.00413509	0.0427842
2  	100   	-0.01	0.0609979  	0.85406  
3  	109   	0.0427842	0.134123   	0.85406  
4  	107   	0.0427842	0.358689   	0.85406  
5  	107   	0.107268 	0.579329   	0.85406  
6  	105   	0.147824 	0.792657   	0.85406  
7  	111   	0.85406  	0.85406    	0.85406  
8  	110   	0.371239 	0.810167   	0.85406  
9  	107   	0.127316 	0.80215    	0.85406  
10 	108   	0.280549 	0.809944   	0.85406  
11 	109   	0.0570733	0.743784   	0.85406  
12 	110   	0.126926 	0.699555   	0.85406  
13 	111   	0.0716033	0.71233    	0.85406  
14 	103   	0.0852458	0.779533   	0.85406  
15 	112   	0.17232  	0.778311   	0.85406  
16 	112   	0.287413 	0.791099   	0.85406  
17 	112   	0.85406  	0.85406    	0.85406  
18 	110   	0.359093 	0.784679   	0.85406  
19 	114   	0.85406  	0.85406    	0.85406  
20 	107   	0.169194 	0.805141   	0.85406  
21 	113   	0.115218 	0.693205   	0.85406  
22 	108   	0.167816 	0.801272   	0.85406  
23 	

### Continual training

# Results